# 03 - MCP 入門：製作你的第一個 MCP Server

**MCP（Model Context Protocol）** 是 Anthropic 制定的開放協定，讓 LLM 應用程式能透過標準化介面連接外部工具與資料。
任何支援 MCP 的 Client（如 Claude Desktop、Cursor）都能自動發現並呼叫你的 Server 所暴露的功能。

一個 MCP Server 可以暴露三種原語（primitives）給 LLM：

| 原語 | 裝飾器 | 用途 |
|------|--------|------|
| **Tools** | `@mcp.tool()` | LLM 主動呼叫的函式（有副作用） |
| **Resources** | `@mcp.resource(uri)` | LLM 可讀取的靜態/動態資料 |
| **Prompts** | `@mcp.prompt()` | 可重用的提示模板 |

## 架構概覽

```
MCP Client
(Claude Desktop / Cursor / 任何相容 App)
        ↕  MCP Protocol（stdio 或 SSE）
MCP Server
(你的 Python server.py)
        ↕
Tools / Resources / Local Data / APIs
```

- **stdio 模式**（預設）：Client 以子行程方式啟動 Server，透過 stdin/stdout 通訊，適合本地工具。
- **SSE 模式**：Server 作為 HTTP 服務運行，適合遠端部署或多 Client 場景。

## 1 · 建立 FastMCP 實例

`FastMCP` 是 MCP Python SDK 提供的高階包裝，讓你用裝飾器直接定義工具，無需手動處理協定細節。

In [13]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-server")
print(f"Server name: {mcp.name}")

Server name: demo-server


## 2 · 定義 Tools

`@mcp.tool()` 將普通 Python 函式包裝成 LLM 可呼叫的工具：
- **docstring** → 工具說明（LLM 據此決定何時呼叫）
- **型別標注** → input schema（SDK 自動透過 Pydantic 生成 JSON Schema）
- **回傳值** → 工具執行結果，傳回給 LLM 繼續推理

In [14]:
@mcp.tool()
def add(a: int, b: int) -> int:
    """將兩個整數相加並回傳結果。"""
    return a + b


@mcp.tool()
def greet(name: str) -> str:
    """以繁體中文問候指定的人。"""
    return f"你好，{name}！歡迎使用 MCP。"

### 直接測試 Tools

Tool 函式本質上就是普通 Python 函式，可以直接在 Notebook 中呼叫，不需要啟動 Server。

In [15]:
print(add(3, 4))
print(greet("世界"))

7
你好，世界！歡迎使用 MCP。


## 3 · 定義 Resources

`@mcp.resource(uri)` 暴露資料給 LLM 讀取。Resources 適合靜態說明文件、設定檔，或動態查詢的資料集。

URI 可以是固定字串（靜態），也可以是帶有參數的模板（動態），例如 `"db://users/{user_id}"`。

In [16]:
@mcp.resource("info://server-description")
def server_description() -> str:
    """這個 demo-server 的簡介與可用工具清單。"""
    return (
        "demo-server 是一個示範用的 MCP Server。\n"
        "可用工具：\n"
        "  - add(a, b)   : 整數加法\n"
        "  - greet(name) : 中文問候\n"
        "可用資源：\n"
        "  - info://server-description : 本說明文字"
    )


# Resource 函式同樣可以直接呼叫
print(server_description())

demo-server 是一個示範用的 MCP Server。
可用工具：
  - add(a, b)   : 整數加法
  - greet(name) : 中文問候
可用資源：
  - info://server-description : 本說明文字


## 4 · 查看已註冊工具的 Schema

MCP SDK 根據函式的型別標注自動生成 JSON Schema，這份 schema 會在 Server 啟動時傳給 Client，讓 LLM 知道每個工具的參數格式。

In [17]:
import inspect

for fn in [add, greet]:
    sig = inspect.signature(fn)
    params = {
        name: str(param.annotation.__name__)
        for name, param in sig.parameters.items()
    }
    print(f"@mcp.tool: {fn.__name__}")
    print(f"  docstring  : {fn.__doc__}")
    print(f"  parameters : {params}")
    print(f"  return type: {sig.return_annotation.__name__}")
    print()

@mcp.tool: add
  docstring  : 將兩個整數相加並回傳結果。
  parameters : {'a': 'int', 'b': 'int'}
  return type: int

@mcp.tool: greet
  docstring  : 以繁體中文問候指定的人。
  parameters : {'name': 'str'}
  return type: str



## 5 · 獨立 Server 檔案（03_mcp_server.py）

MCP Server 需要以獨立的 `.py` 檔執行，透過 `mcp.run()` 以 **stdio 模式**啟動。
此筆記本對應的 server 檔案為 `03_mcp_server.py`（與本 notebook 同目錄）：

In [18]:
import sys
from pathlib import Path

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / "examples"

print((EXAMPLES_DIR / "03_mcp_server.py").read_text())

#!/usr/bin/env python
"""最基礎的 MCP Server 範例 — demo-server。

工具：
  add(a, b)   整數加法
  greet(name) 中文問候

資源：
  info://server-description  本 server 的簡介文字
"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-server")


@mcp.tool()
def add(a: int, b: int) -> int:
    """將兩個整數相加並回傳結果。"""
    return a + b


@mcp.tool()
def greet(name: str) -> str:
    """以繁體中文問候指定的人。"""
    return f"你好，{name}！歡迎使用 MCP。"


@mcp.resource("info://server-description")
def server_description() -> str:
    """這個 demo-server 的簡介與可用工具清單。"""
    return (
        "demo-server 是一個示範用的 MCP Server。\n"
        "可用工具：\n"
        "  - add(a, b)   : 整數加法\n"
        "  - greet(name) : 中文問候\n"
        "可用資源：\n"
        "  - info://server-description : 本說明文字"
    )


if __name__ == "__main__":
    mcp.run()  # 預設以 stdio 模式啟動



### 本地測試（mcp dev）

```bash
# 啟動 MCP Inspector，在瀏覽器中手動呼叫工具
uv run mcp dev examples/03_mcp_server.py
```

MCP Inspector 會在瀏覽器開啟，讓你手動呼叫工具、查看 schema，無需接上真實的 LLM。

## 6 · 使用 LLM 呼叫 MCP Tools

以下示範完整的 **MCP Client ↔ LLM 整合流程**：

```
Notebook（Client）
  1. 以 stdio 啟動 03_mcp_server.py 子行程
  2. 取得 server 的 tool list → 轉成 OpenAI function schema
  3. 把 schema 傳給 LLM → LLM 決定呼叫哪個工具
  4. 透過 MCP 協定執行工具 → 取得結果
  5. 把結果送回 LLM → 取得最終回覆
```

> LLM 設定來自 `.env`（`VLLM_BASE_URL` / `VLLM_API_KEY` / `VLLM_MODEL`）。

In [19]:
import os
import json
import re
from openai import OpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

llm = OpenAI(
    base_url=os.environ["VLLM_BASE_URL"],
    api_key=os.environ["VLLM_API_KEY"],
)
MODEL = os.environ["VLLM_MODEL"]
print(f"LLM: {MODEL}")

LLM: gemma-4-31B-it


### 連線到 MCP Server，列出可用工具

In [20]:
SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,  # 使用目前 venv 的 Python，確保 mcp 套件可被找到
    args=[str(EXAMPLES_DIR / "03_mcp_server.py")],
)


async def list_mcp_tools():
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()
            return result.tools


mcp_tools = await list_mcp_tools()
for t in mcp_tools:
    print(f"  [{t.name}]    {t.description}")

  [add]    將兩個整數相加並回傳結果。
  [greet]    以繁體中文問候指定的人。


### LLM + MCP 整合主函式

`run_with_mcp(question)` 實作完整的單輪工具呼叫循環。
同時處理兩種工具呼叫格式：
- **標準 `tool_calls`**：OpenAI function calling 格式
- **Gemma-4 fallback**：模型有時以純文字輸出 `call:func{key:val}` 格式的工具呼叫

In [21]:
_TEXT_TOOL_RE = re.compile(r"call:(\w+)\{([^}]*)\}")


def _try_cast(v: str):
    """int → float → str の順で型変換を試みる。"""
    try:
        return int(v)
    except ValueError:
        pass
    try:
        return float(v)
    except ValueError:
        pass
    return v


def _parse_all_text_calls(content: str) -> list[tuple[str, dict]]:
    calls = []
    for m in _TEXT_TOOL_RE.finditer(content):
        args = {}
        for kv in m.group(2).split(","):
            if ":" in kv:
                k, v = kv.split(":", 1)
                args[k.strip()] = _try_cast(v.strip())
        calls.append((m.group(1), args))
    return calls


async def run_with_mcp(question: str) -> None:
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Step 1: 取得 MCP tools → 轉成 OpenAI function schema
            result = await session.list_tools()
            openai_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in result.tools
            ]

            messages = [{"role": "user", "content": question}]
            print(f"Q: {question}\n")

            # Step 2: LLM 決定呼叫哪個工具
            resp = llm.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=openai_tools,
                tool_choice="auto",
            )
            msg = resp.choices[0].message

            # Gemma-4 fallback：模型以純文字格式輸出工具呼叫
            if not msg.tool_calls and isinstance(msg.content, str):
                text_calls = _parse_all_text_calls(msg.content)
                if text_calls:
                    tool_results = []
                    for name, args in text_calls:
                        print(f"→ MCP call : {name}({args})")
                        r = await session.call_tool(name, args)
                        out = r.content[0].text if r.content else ""
                        print(f"  MCP result: {out}")
                        tool_results.append(f"{name}({args}) → {out}")
                    messages.append({"role": "assistant", "content": msg.content})
                    messages.append({
                        "role": "user",
                        "content": "工具執行結果：\n" + "\n".join(tool_results) + "\n\n請根據以上結果回答原始問題。",
                    })
                    final = llm.chat.completions.create(model=MODEL, messages=messages)
                    print(f"\nA: {final.choices[0].message.content}")
                    return
                print(f"A: {msg.content}")
                return

            # 標準 tool_calls 路徑
            messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                    }
                    for tc in msg.tool_calls
                ],
            })

            # Step 3: 透過 MCP 協定執行工具
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"→ MCP call : {tc.function.name}({args})")
                r = await session.call_tool(tc.function.name, args)
                out = r.content[0].text if r.content else ""
                print(f"  MCP result: {out}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

            # Step 4: LLM 整合工具結果，給出最終回覆
            final = llm.chat.completions.create(model=MODEL, messages=messages)
            print(f"\nA: {final.choices[0].message.content}")

### 執行示範

In [22]:
# 同時呼叫兩個工具
await run_with_mcp("請幫我計算 123 加 456，並用中文問候『世界』。")

Q: 請幫我計算 123 加 456，並用中文問候『世界』。



HTTP Request: POST http://your-vllm-endpoint:port/v1/chat/completions "HTTP/1.1 200 OK"


→ MCP call : add({'a': 123, 'b': 456})
  MCP result: 579
→ MCP call : greet({'name': '世界'})
  MCP result: 你好，世界！歡迎使用 MCP。


HTTP Request: POST http://your-vllm-endpoint:port/v1/chat/completions "HTTP/1.1 200 OK"



A: 123 加 456 的結果是 579。

你好，世界！


In [23]:
# 只需要加法
await run_with_mcp("999 加 1 等於多少？")

Q: 999 加 1 等於多少？



HTTP Request: POST http://your-vllm-endpoint:port/v1/chat/completions "HTTP/1.1 200 OK"


→ MCP call : add({'a': 999, 'b': 1})
  MCP result: 1000


HTTP Request: POST http://your-vllm-endpoint:port/v1/chat/completions "HTTP/1.1 200 OK"



A: 999 加 1 等於 1000。


## 小結

本筆記本涵蓋了 MCP Server 的完整開發與整合流程：

1. `uv pip install mcp` — 安裝 SDK
2. `FastMCP(name)` — 建立 server 實例
3. `@mcp.tool()` — 定義 LLM 可呼叫的函式（docstring + type hints 自動生成 schema）
4. `@mcp.resource(uri)` — 暴露資料給 LLM 讀取
5. `mcp.run()` — 以 stdio 模式啟動 server（獨立 `03_mcp_server.py`）
6. `ClientSession` + `stdio_client` — 從 Python 以 MCP 協定連線至 server
7. LLM 呼叫流程：取得 tool schema → 送給 LLM → 執行 MCP tool → 回傳結果 → 最終回覆

---

**下一步可以探索：**
- `uv run mcp dev examples/03_mcp_server.py` — 開啟 MCP Inspector 進行互動測試
- `mcp.run(transport="sse")` — 改為 SSE 模式，提供 HTTP 端點
- `@mcp.prompt()` — 定義可重用的 prompt 模板
- 動態 resource URI 模板：`@mcp.resource("db://users/{user_id}")`
- 多輪對話 + 多次工具呼叫（LangGraph ReAct loop）